# SCE-Net для compatibility-задачи (cosine version, starting from ready triplets)

Этот ноутбук **начинается с момента, когда у вас уже есть**:
- `train_triplets`
- `val_triplets`

То есть здесь **нет** повторной загрузки `pairs.csv`, split-а и построения триплетов. Мы сразу стартуем с обучения.

## Что должно быть подготовлено заранее

### Обязательные датафреймы
- `train_triplets`
- `val_triplets`

Они должны содержать как минимум колонки:
- `anchor_path`
- `pos_path`
- `neg_path`

### Необязательные датафреймы для pair-level evaluation
Если хотите считать ROC-AUC / PR-AUC на обычных парах, можно заранее иметь один из наборов:
- `val_pairs_df` / `test_pairs_df`, либо
- `val_df` / `test_df`

Для pair-level evaluation ожидаются колонки:
- `sku1_path`
- `sku2_path`
- `y`

---

## Что меняется по сравнению с L2-версией

- обучение переводится с Euclidean triplet loss на **cosine triplet loss**;
- conditioned embeddings нормализуются перед сравнением;
- для FashionCLIP добавлено **частичное fine-tuning**, а не грубое размораживание всего backbone;
- остальная логика SCE-Net максимально переиспользуется.

Итого: да, **обучаться на cosine здесь можно и логично**, особенно для CLIP-подобных эмбеддингов.


## 1) Импорты

In [ ]:
# !pip install -q torch torchvision transformers pandas scikit-learn opencv-python-headless tqdm matplotlib

import random
import time
from dataclasses import dataclass
from typing import Optional, Tuple

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, average_precision_score
from transformers import CLIPModel, CLIPProcessor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


## 2) Конфигурация

Ключевые отличия от старой версии:
- `loss_metric='cosine'`
- `default_score_mode='cosine'`
- `freeze_backbone=False`, но при этом размораживаются **не все слои**, а только верхушка vision-backbone.

Рекомендуемый старт:
- последние `2` vision block-а,
- `visual_projection`,
- norm-слои vision-ветки,
- маленький `backbone_lr`.


In [ ]:
@dataclass
class Config:
    hf_model_name: str = 'patrickjohncyh/fashion-clip'
    num_conditions: int = 5
    condition_hidden: int = 1024
    dropout: float = 0.1

    batch_size: int = 64
    num_workers: int = 8
    prefetch_factor: int = 4
    persistent_workers: bool = True

    head_lr: float = 1e-4
    backbone_lr: float = 2e-6
    weight_decay: float = 1e-5
    epochs: int = 8
    margin: float = 0.1
    lambda_l1: float = 1e-5
    lambda_l2: float = 1e-4
    grad_clip_norm: float = 1.0

    loss_metric: str = 'cosine'
    default_score_mode: str = 'cosine'

    freeze_backbone: bool = False
    trainable_vision_blocks: int = 2
    unfreeze_visual_projection: bool = True
    unfreeze_vision_norms: bool = True
    unfreeze_vision_embeddings: bool = False

    use_amp: bool = True

cfg = Config()
cfg


## 3) Проверяем, что готовые triplets уже существуют

In [ ]:
REQUIRED_TRIPLET_COLS = {'anchor_path', 'pos_path', 'neg_path'}
REQUIRED_PAIR_COLS = {'sku1_path', 'sku2_path', 'y'}


def validate_triplet_df(name: str, df: pd.DataFrame):
    if name not in globals():
        raise NameError(f'{name} is not defined. Prepare it before running this notebook.')
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f'{name} must be a pandas DataFrame.')
    missing = REQUIRED_TRIPLET_COLS - set(df.columns)
    if missing:
        raise ValueError(f'{name} is missing columns: {missing}')
    if len(df) == 0:
        raise ValueError(f'{name} is empty.')


def validate_pair_df(name: str, df: pd.DataFrame):
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f'{name} must be a pandas DataFrame.')
    missing = REQUIRED_PAIR_COLS - set(df.columns)
    if missing:
        raise ValueError(f'{name} is missing columns: {missing}')


def resolve_pair_df(split: str) -> Tuple[Optional[pd.DataFrame], Optional[str]]:
    for candidate in (f'{split}_pairs_df', f'{split}_df'):
        obj = globals().get(candidate)
        if isinstance(obj, pd.DataFrame):
            return obj, candidate
    return None, None


validate_triplet_df('train_triplets', train_triplets)
validate_triplet_df('val_triplets', val_triplets)
print('train_triplets:', train_triplets.shape)
print('val_triplets:', val_triplets.shape)

val_pair_df, val_pair_name = resolve_pair_df('val')
test_pair_df, test_pair_name = resolve_pair_df('test')

if val_pair_df is not None:
    validate_pair_df(val_pair_name, val_pair_df)
    print(f'Found validation pair dataframe: {val_pair_name} -> {val_pair_df.shape}')
else:
    print('Validation pair dataframe not found. Pair-level ROC-AUC/PR-AUC block will be skipped.')

if test_pair_df is not None:
    validate_pair_df(test_pair_name, test_pair_df)
    print(f'Found test pair dataframe: {test_pair_name} -> {test_pair_df.shape}')
else:
    print('Test pair dataframe not found. Test pair-level evaluation block will be skipped.')


## 4) Dataset / DataLoader и RAM cache

In [ ]:
def collect_image_paths(train_triplets: pd.DataFrame, val_triplets: pd.DataFrame, *pair_dfs: pd.DataFrame):
    paths = []
    for df in [train_triplets, val_triplets]:
        paths.extend(df['anchor_path'].astype(str).tolist())
        paths.extend(df['pos_path'].astype(str).tolist())
        paths.extend(df['neg_path'].astype(str).tolist())

    for pair_df in pair_dfs:
        if pair_df is not None:
            paths.extend(pair_df['sku1_path'].astype(str).tolist())
            paths.extend(pair_df['sku2_path'].astype(str).tolist())

    return sorted(set(paths))


def build_image_cache(paths, verbose: bool = True):
    unique_paths = sorted(set(map(str, paths)))
    cache = {}

    iterator = tqdm(unique_paths, desc='Caching images to RAM') if verbose else unique_paths
    for p in iterator:
        img_bgr = cv2.imread(p, cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise FileNotFoundError(f'Cannot read image while caching: {p}')
        cache[p] = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    if verbose:
        total_mb = sum(v.nbytes for v in cache.values()) / (1024 ** 2)
        print(f'Cached images: {len(cache)} | approx RAM: {total_mb:.1f} MB')
    return cache


class TripletImageDataset(Dataset):
    def __init__(self, triplet_df: pd.DataFrame, processor: CLIPProcessor, image_cache: dict):
        self.df = triplet_df.reset_index(drop=True)
        self.processor = processor
        self.image_cache = image_cache

    def __len__(self):
        return len(self.df)

    def _load_img(self, p: str):
        p = str(p)
        if p not in self.image_cache:
            raise KeyError(f'Image path not found in RAM cache: {p}')
        return self.image_cache[p]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return self._load_img(row['anchor_path']), self._load_img(row['pos_path']), self._load_img(row['neg_path'])


def make_triplet_collate(processor: CLIPProcessor):
    def collate_fn(batch):
        a_imgs, p_imgs, n_imgs = zip(*batch)
        a_inputs = processor(images=list(a_imgs), return_tensors='pt')
        p_inputs = processor(images=list(p_imgs), return_tensors='pt')
        n_inputs = processor(images=list(n_imgs), return_tensors='pt')
        return a_inputs['pixel_values'], p_inputs['pixel_values'], n_inputs['pixel_values']

    return collate_fn


## 5) Реализация SCE-Net

In [ ]:
class SCENet(nn.Module):
    def __init__(self, clip_model_name: str, num_conditions: int = 5, hidden_dim: int = 1024, dropout: float = 0.1):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(clip_model_name)

        if hasattr(self.clip, 'visual_projection') and self.clip.visual_projection is not None:
            proj = self.clip.visual_projection
            emb_dim = proj.out_features if hasattr(proj, 'out_features') else self.clip.config.projection_dim
        else:
            emb_dim = getattr(self.clip.config, 'projection_dim', None)
            if emb_dim is None:
                raise ValueError('Cannot infer embedding dimension D from model config/projection.')

        self.emb_dim = emb_dim
        self.num_conditions = num_conditions

        self.condition_masks = nn.Parameter(torch.empty(num_conditions, emb_dim))
        nn.init.xavier_uniform_(self.condition_masks)

        self.weight_branch = nn.Sequential(
            nn.Linear(2 * emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_conditions),
        )

    def _extract_image_features(self, pixel_values: torch.Tensor) -> torch.Tensor:
        if hasattr(self.clip, 'get_image_features'):
            out = self.clip.get_image_features(pixel_values=pixel_values)
            if torch.is_tensor(out):
                return out
            if hasattr(out, 'image_embeds') and out.image_embeds is not None:
                return out.image_embeds
            if hasattr(out, 'pooler_output') and out.pooler_output is not None:
                return out.pooler_output
            if hasattr(out, 'last_hidden_state') and out.last_hidden_state is not None:
                return out.last_hidden_state[:, 0, :]

        out = self.clip(pixel_values=pixel_values, return_dict=True)
        if hasattr(out, 'image_embeds') and out.image_embeds is not None:
            return out.image_embeds
        if hasattr(out, 'pooler_output') and out.pooler_output is not None:
            return out.pooler_output
        if hasattr(out, 'last_hidden_state') and out.last_hidden_state is not None:
            return out.last_hidden_state[:, 0, :]

        raise TypeError(f'Unsupported image output type: {type(out)}')

    def encode_image(self, pixel_values: torch.Tensor) -> torch.Tensor:
        img_feat = self._extract_image_features(pixel_values)
        return F.normalize(img_feat, p=2, dim=-1)

    def compute_w(self, v1: torch.Tensor, v2: torch.Tensor) -> torch.Tensor:
        logits = self.weight_branch(torch.cat([v1, v2], dim=-1))
        return F.softmax(logits, dim=-1)

    def apply_conditions(self, v: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
        masked = v.unsqueeze(1) * self.condition_masks.unsqueeze(0)
        return torch.einsum('bm,bmd->bd', w, masked)

    def project_pair_from_embeddings(self, v1: torch.Tensor, v2: torch.Tensor):
        w = self.compute_w(v1, v2)
        e1 = self.apply_conditions(v1, w)
        e2 = self.apply_conditions(v2, w)
        return e1, e2, w

    def forward_pair(self, img1: torch.Tensor, img2: torch.Tensor):
        v1 = self.encode_image(img1)
        v2 = self.encode_image(img2)
        e1, e2, w = self.project_pair_from_embeddings(v1, v2)
        return e1, e2, w, v1, v2

    def forward_triplet(self, a_px: torch.Tensor, p_px: torch.Tensor, n_px: torch.Tensor):
        v_a = self.encode_image(a_px)
        v_p = self.encode_image(p_px)
        v_n = self.encode_image(n_px)

        e_a_pos, e_p, w_ap = self.project_pair_from_embeddings(v_a, v_p)
        e_a_neg, e_n, w_an = self.project_pair_from_embeddings(v_a, v_n)

        e_a = 0.5 * (e_a_pos + e_a_neg)
        return e_a, e_p, e_n, w_ap, w_an


## 6) Cosine loss и частичное дообучение FashionCLIP

Здесь ключевая идея такая:
- после condition masks и агрегации масштаб эмбеддингов уже не гарантированно стабилен;
- поэтому перед cosine-сравнением мы **нормализуем итоговые conditioned embeddings**;
- triplet hinge строим уже на `pos_sim` и `neg_sim`.

Fine-tuning лучше начинать не с полного размораживания модели, а с:
- `visual_projection`
- norm-слоёв
- последних `1-2` vision-блоков

Это безопаснее и обычно лучше переносит предобучение FashionCLIP в задачу compatibility.


In [ ]:
def compute_similarity_scores(e1, e2, score_mode: str = 'cosine', temperature: float = 1.0):
    if temperature <= 0:
        raise ValueError('temperature must be > 0')

    e1n = F.normalize(e1, p=2, dim=-1)
    e2n = F.normalize(e2, p=2, dim=-1)
    cosine = (e1n * e2n).sum(dim=-1)

    if score_mode == 'cosine':
        return cosine
    if score_mode == 'cosine_01':
        return (cosine + 1.0) / 2.0
    if score_mode == 'neg_l2':
        return -torch.norm(e1n - e2n, p=2, dim=-1)
    if score_mode == 'prob':
        dist = torch.norm(e1n - e2n, p=2, dim=-1)
        return torch.sigmoid(-dist / temperature)

    raise ValueError(f'Unknown score_mode={score_mode}')


def sce_loss(e_anchor, e_pos, e_neg, condition_masks, margin, lambda_l1, lambda_l2, metric: str = 'cosine'):
    raw_anchor, raw_pos, raw_neg = e_anchor, e_pos, e_neg

    if metric == 'cosine':
        e_anchor = F.normalize(e_anchor, p=2, dim=-1)
        e_pos = F.normalize(e_pos, p=2, dim=-1)
        e_neg = F.normalize(e_neg, p=2, dim=-1)

        pos_sim = F.cosine_similarity(e_anchor, e_pos, dim=-1)
        neg_sim = F.cosine_similarity(e_anchor, e_neg, dim=-1)
        triplet = F.relu(neg_sim - pos_sim + margin).mean()

        stats = {
            'pos_sim': pos_sim.mean().item(),
            'neg_sim': neg_sim.mean().item(),
        }
    elif metric == 'l2':
        d_pos = torch.norm(e_anchor - e_pos, p=2, dim=-1)
        d_neg = torch.norm(e_anchor - e_neg, p=2, dim=-1)
        triplet = F.relu(d_pos - d_neg + margin).mean()
        stats = {
            'd_pos': d_pos.mean().item(),
            'd_neg': d_neg.mean().item(),
        }
    else:
        raise ValueError(f'Unsupported metric={metric}')

    l1 = condition_masks.abs().mean()
    l2 = (raw_anchor.pow(2).mean() + raw_pos.pow(2).mean() + raw_neg.pow(2).mean()) / 3.0

    total = triplet + lambda_l1 * l1 + lambda_l2 * l2
    stats.update({
        'loss': total.item(),
        'triplet': triplet.item(),
        'l1': l1.item(),
        'l2': l2.item(),
    })
    return total, stats


def set_module_requires_grad(module: nn.Module, requires_grad: bool):
    if module is None:
        return
    for p in module.parameters():
        p.requires_grad = requires_grad


def get_vision_encoder_layers(clip_model: CLIPModel):
    vision_model = getattr(clip_model, 'vision_model', None)
    if vision_model is None:
        return []

    encoder = getattr(vision_model, 'encoder', None)
    if encoder is None:
        return []

    layers = getattr(encoder, 'layers', None)
    if layers is None:
        layers = getattr(encoder, 'layer', None)
    if layers is None:
        return []

    return list(layers)


def configure_partial_finetuning(model: nn.Module, cfg: Config):
    for p in model.clip.parameters():
        p.requires_grad = False

    trainable_groups = []

    if cfg.freeze_backbone:
        return ['head_only']

    if cfg.unfreeze_visual_projection and hasattr(model.clip, 'visual_projection'):
        set_module_requires_grad(model.clip.visual_projection, True)
        trainable_groups.append('visual_projection')

    vision_model = getattr(model.clip, 'vision_model', None)
    if vision_model is not None and cfg.unfreeze_vision_norms:
        for attr in ['pre_layrnorm', 'pre_layernorm', 'post_layernorm']:
            if hasattr(vision_model, attr):
                set_module_requires_grad(getattr(vision_model, attr), True)
                trainable_groups.append(attr)

    if vision_model is not None and cfg.unfreeze_vision_embeddings and hasattr(vision_model, 'embeddings'):
        set_module_requires_grad(vision_model.embeddings, True)
        trainable_groups.append('vision_embeddings')

    layers = get_vision_encoder_layers(model.clip)
    if cfg.trainable_vision_blocks > 0 and layers:
        blocks_to_unfreeze = layers[-cfg.trainable_vision_blocks:]
        for idx, block in enumerate(blocks_to_unfreeze, start=len(layers) - len(blocks_to_unfreeze)):
            set_module_requires_grad(block, True)
            trainable_groups.append(f'vision_block_{idx}')

    return trainable_groups


def build_optimizer(model: nn.Module, cfg: Config):
    param_groups = {
        'head_decay': {'params': [], 'lr': cfg.head_lr, 'weight_decay': cfg.weight_decay},
        'head_no_decay': {'params': [], 'lr': cfg.head_lr, 'weight_decay': 0.0},
        'backbone_decay': {'params': [], 'lr': cfg.backbone_lr, 'weight_decay': cfg.weight_decay},
        'backbone_no_decay': {'params': [], 'lr': cfg.backbone_lr, 'weight_decay': 0.0},
    }

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue

        is_backbone = name.startswith('clip.')
        no_decay = (p.ndim < 2) or name.endswith('.bias') or ('norm' in name.lower())

        if is_backbone and no_decay:
            param_groups['backbone_no_decay']['params'].append(p)
        elif is_backbone:
            param_groups['backbone_decay']['params'].append(p)
        elif no_decay:
            param_groups['head_no_decay']['params'].append(p)
        else:
            param_groups['head_decay']['params'].append(p)

    groups = [group for group in param_groups.values() if group['params']]
    return torch.optim.AdamW(groups)


def summarize_trainable_parameters(model: nn.Module, max_rows: int = 30):
    rows = []
    for name, p in model.named_parameters():
        if p.requires_grad:
            rows.append((name, p.numel()))

    rows = sorted(rows, key=lambda x: x[0])
    print('Trainable parameter tensors:', len(rows))
    for name, n in rows[:max_rows]:
        print(f'  {name}: {n:,}')
    if len(rows) > max_rows:
        print(f'  ... and {len(rows) - max_rows} more')


## 7) Инициализация обучения — теперь прямо с готовых triplets

In [ ]:
processor = CLIPProcessor.from_pretrained(cfg.hf_model_name)

all_paths = collect_image_paths(train_triplets, val_triplets, val_pair_df, test_pair_df)
IMAGE_CACHE = build_image_cache(all_paths, verbose=True)

train_ds = TripletImageDataset(train_triplets, processor, image_cache=IMAGE_CACHE)
val_ds = TripletImageDataset(val_triplets, processor, image_cache=IMAGE_CACHE)
collate_fn = make_triplet_collate(processor)

loader_kwargs = dict(
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn,
)
if cfg.num_workers > 0:
    loader_kwargs['prefetch_factor'] = cfg.prefetch_factor
    loader_kwargs['persistent_workers'] = cfg.persistent_workers

train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)

model = SCENet(cfg.hf_model_name, cfg.num_conditions, cfg.condition_hidden, cfg.dropout).to(device)
trainable_groups = configure_partial_finetuning(model, cfg)
optimizer = build_optimizer(model, cfg)
scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and torch.cuda.is_available()))

print('loss metric:', cfg.loss_metric)
print('default score mode:', cfg.default_score_mode)
print('trainable backbone groups:', trainable_groups)
print('trainable params:', sum(p.numel() for p in model.parameters() if p.requires_grad))
print('embedding dim D:', model.emb_dim)
summarize_trainable_parameters(model)


## 8) Train / Val loop

In [ ]:
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    losses, triplets = [], []
    y_true, y_score = [], []
    pos_sims, neg_sims = [], []

    data_time = 0.0
    compute_time = 0.0
    t_prev = time.perf_counter()

    for a_px, p_px, n_px in tqdm(loader, leave=False):
        t_after_load = time.perf_counter()
        data_time += (t_after_load - t_prev)

        a_px = a_px.to(device, non_blocking=True)
        p_px = p_px.to(device, non_blocking=True)
        n_px = n_px.to(device, non_blocking=True)

        t0 = time.perf_counter()
        with torch.set_grad_enabled(is_train):
            with torch.cuda.amp.autocast(enabled=(cfg.use_amp and torch.cuda.is_available())):
                e_a, e_p, e_n, _, _ = model.forward_triplet(a_px, p_px, n_px)
                loss, stats = sce_loss(
                    e_a,
                    e_p,
                    e_n,
                    model.condition_masks,
                    cfg.margin,
                    cfg.lambda_l1,
                    cfg.lambda_l2,
                    metric=cfg.loss_metric,
                )

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                if cfg.grad_clip_norm is not None:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
                scaler.step(optimizer)
                scaler.update()

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        compute_time += (time.perf_counter() - t0)

        losses.append(stats['loss'])
        triplets.append(stats['triplet'])

        ap_score = compute_similarity_scores(e_a, e_p, score_mode=cfg.default_score_mode).detach().cpu().numpy()
        an_score = compute_similarity_scores(e_a, e_n, score_mode=cfg.default_score_mode).detach().cpu().numpy()
        y_score.extend(ap_score.tolist())
        y_score.extend(an_score.tolist())
        y_true.extend([1] * len(ap_score))
        y_true.extend([0] * len(an_score))

        if 'pos_sim' in stats:
            pos_sims.append(stats['pos_sim'])
        if 'neg_sim' in stats:
            neg_sims.append(stats['neg_sim'])

        t_prev = time.perf_counter()

    out = {
        'loss': float(np.mean(losses)) if losses else np.nan,
        'triplet': float(np.mean(triplets)) if triplets else np.nan,
        'data_time_s': float(data_time),
        'compute_time_s': float(compute_time),
    }

    if pos_sims:
        out['pos_sim'] = float(np.mean(pos_sims))
    if neg_sims:
        out['neg_sim'] = float(np.mean(neg_sims))

    if len(set(y_true)) == 2:
        out['triplet_pair_auc'] = float(roc_auc_score(y_true, y_score))
    else:
        out['triplet_pair_auc'] = np.nan

    return out


best_val = float('inf')
best_path = 'sce_net_cosine_best.pt'
history = []

for epoch in range(1, cfg.epochs + 1):
    train_metrics = run_epoch(model, train_loader, optimizer=optimizer)
    val_metrics = run_epoch(model, val_loader, optimizer=None)

    history.append({
        'epoch': epoch,
        **{f'train_{k}': v for k, v in train_metrics.items()},
        **{f'val_{k}': v for k, v in val_metrics.items()},
    })

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_metrics['loss']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"train_triplet_auc={train_metrics['triplet_pair_auc']:.4f} | "
        f"val_triplet_auc={val_metrics['triplet_pair_auc']:.4f} | "
        f"train_pos_sim={train_metrics.get('pos_sim', float('nan')):.4f} | "
        f"train_neg_sim={train_metrics.get('neg_sim', float('nan')):.4f} | "
        f"train(data/compute)={train_metrics['data_time_s']:.1f}/{train_metrics['compute_time_s']:.1f}s"
    )

    if val_metrics['loss'] < best_val:
        best_val = val_metrics['loss']
        torch.save({'model_state': model.state_dict(), 'config': cfg.__dict__}, best_path)
        print('  -> saved best checkpoint:', best_path)

pd.DataFrame(history)


## 9) Optional pair-level evaluation

In [ ]:
@torch.no_grad()
def pair_scores(
    model: SCENet,
    pair_df: pd.DataFrame,
    processor: CLIPProcessor,
    batch_size: int = 64,
    image_cache: dict = None,
    score_mode: str = 'cosine',
    temperature: float = 1.0,
):
    model.eval()
    scores = []
    labels = pair_df['y'].astype(int).tolist()

    rows = pair_df[['sku1_path', 'sku2_path']].astype(str).values.tolist()
    for i in tqdm(range(0, len(rows), batch_size)):
        chunk = rows[i:i + batch_size]
        imgs1, imgs2 = [], []
        for a, b in chunk:
            if image_cache is not None:
                if a not in image_cache or b not in image_cache:
                    raise KeyError(f'Pair path not found in cache: {a}, {b}')
                imgs1.append(image_cache[a])
                imgs2.append(image_cache[b])
            else:
                a_bgr = cv2.imread(a, cv2.IMREAD_COLOR)
                b_bgr = cv2.imread(b, cv2.IMREAD_COLOR)
                if a_bgr is None or b_bgr is None:
                    raise FileNotFoundError(f'Cannot read image in pair: {a}, {b}')
                imgs1.append(cv2.cvtColor(a_bgr, cv2.COLOR_BGR2RGB))
                imgs2.append(cv2.cvtColor(b_bgr, cv2.COLOR_BGR2RGB))

        px1 = processor(images=imgs1, return_tensors='pt')['pixel_values'].to(device)
        px2 = processor(images=imgs2, return_tensors='pt')['pixel_values'].to(device)

        e1, e2, _, _, _ = model.forward_pair(px1, px2)
        batch_scores = compute_similarity_scores(e1, e2, score_mode=score_mode, temperature=temperature)
        scores.extend(batch_scores.cpu().numpy().tolist())

    return np.array(scores), np.array(labels)


def evaluate_pairs_if_available(split_name: str, pair_df: Optional[pd.DataFrame], pair_df_name: Optional[str]):
    if pair_df is None:
        print(f'Skip {split_name}: pair dataframe is not available.')
        return

    scores_cos, labels = pair_scores(model, pair_df, processor, image_cache=IMAGE_CACHE, score_mode='cosine')
    scores_cos01, _ = pair_scores(model, pair_df, processor, image_cache=IMAGE_CACHE, score_mode='cosine_01')
    scores_l2, _ = pair_scores(model, pair_df, processor, image_cache=IMAGE_CACHE, score_mode='neg_l2')

    print(f'[{split_name}] source dataframe: {pair_df_name}')
    print(f'[{split_name}][cosine] ROC-AUC={roc_auc_score(labels, scores_cos):.4f} PR-AUC={average_precision_score(labels, scores_cos):.4f}')
    print(f'[{split_name}][cosine_01] ROC-AUC={roc_auc_score(labels, scores_cos01):.4f} PR-AUC={average_precision_score(labels, scores_cos01):.4f}')
    print(f'[{split_name}][neg_l2] ROC-AUC={roc_auc_score(labels, scores_l2):.4f} PR-AUC={average_precision_score(labels, scores_l2):.4f}')


ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt['model_state'])

evaluate_pairs_if_available('val', val_pair_df, val_pair_name)
evaluate_pairs_if_available('test', test_pair_df, test_pair_name)


## 10) Практические рекомендации по fine-tuning

### Можно ли заменить L2 на cosine?
Да. Для FashionCLIP это хороший и естественный вариант, потому что CLIP-подобные модели обычно лучше ведут себя в угловой геометрии, чем в сырой евклидовой.

### Нужны ли нормализованные эмбеддинги?
Да. Важно нормализовать именно **conditioned embeddings после masks и weight aggregation**, а не только исходный backbone-output.

### Нужно ли ставить `freeze_backbone=False`?
Да, можно, но **не стоит обучать все слои сразу**.

Лучший стартовый рецепт обычно такой:
1. разморозить `visual_projection`;
2. разморозить norm-слои vision-модели;
3. разморозить последние `1-2` transformer block-а;
4. держать `backbone_lr` в `10x-50x` раз меньше, чем `head_lr`.

### Когда увеличивать число размороженных блоков?
- если данных много;
- если train loss падает, а val не улучшается слишком медленно;
- если видно, что head уже адаптировался, а backbone ещё недоучен под вашу compatibility-задачу.

### Когда не стоит размораживать больше слоёв?
- если датасет маленький;
- если val-метрики начинают деградировать;
- если переобучение начинается очень рано.

### Итого
Если кратко: для вашей задачи я бы начинал **с cosine-loss + нормализованных conditioned embeddings + partial fine-tuning последних 2 блоков FashionCLIP**, а не с полного обучения всего backbone.
